In [82]:
import torch
from torch.nn.functional import conv2d, conv3d
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import convolve, generate_binary_structure

#torch.set_default_device(device='cuda')
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
device

device(type='cuda')

Lo que ahora quiero hacer es poder hacer varias simulaciones en N dimensiones en paralelo por lo que voy a apilar lattices en un mismo tensor

In [83]:
T = 0.5 # K
kB = 1 # J/K
beta = 1/(kB*T)

N_lattices = 3
dim_lattices = [10, 10] # todas las lattices generadas deben tener misma dim para correr en paralelo
up_spins = [0.5, 0.5, 0.5]

lattices_list = [] # sublistas: modelos [dim1, dim2, dim3, fracción spin up inicial]
for j in range(0, N_lattices, 1):
    dim_lat = dim_lattices.copy()
    dim_lat.append(up_spins[j])
    lattices_list.append(dim_lat)

lattices = []
for dims in lattices_list:
    ups = dims.pop()
    lattice = np.random.choice([1, -1], size = tuple(dims), p=[ups, 1-ups])
    lattices.append(lattice)
lattices = torch.tensor(np.array(lattices)).to(device)

if not len(lattice[0])>3:
    plt.imshow(lattices[0].cpu())

print('Lattices shape: ', lattices.shape)

Lattices shape:  torch.Size([3, 10, 10])


In [ ]:
def getEnergy(lattices, J = 1):
    LatShiftedUp =torch.roll(lattices, shifts=1, dims= 1)
    LatShiftedRight = torch.roll(lattices, shifts=1, dims=2)
    
    UpEnergyMat = lattices*LatShiftedUp
    RightEnergyMat = lattices*LatShiftedRight

    if len(lattice.shape)==3:
        LatShiftedForward = torch.roll(lattices, shifts=1, dims=3)
        ForwardEnergyMat = lattices*LatShiftedForward
        Einit = -1*J*(torch.sum(UpEnergyMat, dim=(1, 2, 3))+
                      torch.sum(RightEnergyMat, dim=(1, 2, 3))+
                      torch.sum(ForwardEnergyMat, dim=(1, 2, 3)))
    else:
        Einit = -1*J*(torch.sum(UpEnergyMat, dim=(1, 2))+
                      torch.sum(RightEnergyMat, dim=(1, 2)))
        
    return Einit

def getEnergyLattice(lattices, J=1): # nos devuevle tensor de shape igual que lattices donde cada elemento es la energía calcualda con sus vecinos
    if len(lattices.shape)==3:
        mask2d = torch.zeros((3, 3), dtype=bool).to(device)
        mask2d[0, 1] = mask2d[2, 1] = mask2d[1, 0] = mask2d[1, 2] = 1
        mask2d = mask2d.unsqueeze(0).unsqueeze(0).to(device)
        lattices = lattices.unsqueeze(1) # (Nº redes, 1 canal, altura, anchura)
        Etotal = -J*lattices*conv2d(input=lattices.to(torch.float32), weight=mask2d.to(torch.float32), padding='same') #padding para que no se coma el borde
        return Etotal
    
    else:
        mask3d = torch.zeros((3, 3, 3), dtype=bool).to(device)
        mask3d[0, 1, 1] = mask3d[2, 1, 1] = mask3d[1, 0, 1] = mask3d[1, 1, 0] = mask3d[1, 1, 2] = mask3d[1, 2, 1] = 1
        mask3d = mask3d.unsqueeze(0).unsqueeze(0).to(device) # (1 output, 1 canal, altura, anchura, profundidad)
        lattices = lattices.unsqueeze(1) # (Nº redes, 1 canal, altura, anchura, profundidad)
        #print(mask3d.shape, lattices.shape)
        Etotal = -J*lattices*conv3d(input=lattices.to(torch.float32), weight=mask3d.to(torch.float32), padding='same') #padding para que no se coma el borde
        return Etotal


def MetropolisStep(lattices, J=1):
    random_flat_idx = torch.randint(low=0, high=np.prod(dim_lattices), size=(1,))
    idx_element = torch.unravel_index(random_flat_idx, shape=tuple(dim_lattices))
        
def getEnergyScalar(lattices):
    return getEnergyLattice(lattices).sum(axis=(2, 3)).squeeze()

def getDeltaEnergyLattice(lattices):
    return -2*getEnergyLattice(lattices)

getEnergyLattice(lattices).shape
#getNeighboursEnergy(lattices)


torch.Size([3, 1, 10, 10])

In [117]:
random_flat_idx = torch.randint(low=0, high=np.prod(dim_lattices), size=(1,))
idx_element = torch.unravel_index(random_flat_idx, shape=tuple(dim_lattices))
idx_element


(tensor([2], device='cuda:0'), tensor([7], device='cuda:0'))